## Black-Scholes with Put and Call outputs

In this notebook we will use the option pricing dataset generated in the notebook <code>Generate Option Training Data.ipynb</code> to train a Neural Network model that outputs both put and call, and compare that to models trained to just output call prices or put prices

In [ ]:
import pandas as pd
import numpy as np
import torch
from torchvision import datasets, transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import trange
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Load Data

The data was saved to a csv file and so this is loaded into a Pandas dataframe.

In [ ]:
filename = 'OptionData.csv'
df = pd.read_csv(filename)
df.head()

### Split Data

The data can now be split into test and training sets. Given the data was randomly generated and that each row of the dataframe contains both put and call we can simply slide the dataframe. We will used 20,000 rows as the test set.

In [ ]:
train_df = df[:-20000]
test_df = df[-20000:]

### Prepare Data

The option data from the file is not in the right format for training. In particular all the data for a call and a put with the same inputs are on the same row of the dataframe and the frame contains the option "Greeks". The model will be trained to price both puts and calls so we need to split the data and add a binary input flag to show whether the example is for a put or call. Finally we need to split the input data (X) from the labels (y), the option value. The following function contains the all the steps that are required.

In [ ]:
def prepareDataSingle(data_df, call=True):
    drop_list = ['CallValue', 'CallDelta', 
                'CallGamma', 'CallTheta', 'CallVega', 
                'CallRho', 'PutValue', 'PutDelta', 
                'PutGamma', 'PutTheta', 'PutVega', 'PutRho']

    input_data = data_df.copy(deep=True)
    input_data = input_data.drop(drop_list, axis=1)
    
    call_input_data = input_data.copy(deep=True)
    put_input_data = input_data.copy(deep=True)
    
    if call:
        X = call_input_data
        y = data_df['CallValue']
    else:
        X = put_input_data
        y = data_df['PutValue']
    
    return X, y

In [ ]:
def prepareDataBoth(data_df):
    drop_list = ['CallValue', 'CallDelta', 
                'CallGamma', 'CallTheta', 'CallVega', 
                'CallRho', 'PutValue', 'PutDelta', 
                'PutGamma', 'PutTheta', 'PutVega', 'PutRho']

    input_data = data_df.copy(deep=True)
    input_data = input_data.drop(drop_list, axis=1)
    
    call_input_data = input_data.copy(deep=True)
    put_input_data = input_data.copy(deep=True)
    
    X = call_input_data
    y = data_df[['CallValue', 'PutValue']]

    return X, y

In [ ]:
no_epochs = 50
batch_size = 1024

### Build Model

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self, inputs, outputs):
        super(NeuralNetwork, self).__init__()
        self.fc1 = nn.Linear(inputs, 50)
        self.fc2 = nn.Linear(50, 50)
        self.fc3 = nn.Linear(50, outputs)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs):
    train_errors = []
    test_errors = []

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0

        # Training
        for batch_X, batch_y in train_loader:
            # Forward pass
            outputs = model(batch_X)
            loss = loss_fn(outputs.squeeze(), batch_y)
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * batch_X.size(0)

        train_loss /= len(train_loader.dataset)
        train_errors.append(train_loss)
        
        # Evaluation on test set
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                outputs = model(batch_X)
                loss = loss_fn(outputs.squeeze(), batch_y)
                test_loss += loss.item() * batch_X.size(0)

        test_loss /= len(test_loader.dataset)
        test_errors.append(test_loss)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} \
            - Train loss: {train_loss:.4f}, Test loss: {test_loss:.4f}")

    history = dict()
    history['train_loss'] = train_errors
    history['test_loss'] = test_errors
    return history

In [ ]:
history_dict = dict()

### Call Option

In [ ]:
X, y = prepareDataSingle(train_df, True)

In [ ]:
X_test, y_test = prepareDataSingle(test_df, True)

In [ ]:
standard_scalar = StandardScaler()
X[['Vol', 'Spot', 'T']] = standard_scalar.fit_transform(X[['Vol', 'Spot', 'T']])

In [ ]:
X_test[['Vol', 'Spot', 'T']] = standard_scalar.transform(X_test[['Vol', 'Spot', 'T']])

In [ ]:
train_x = torch.Tensor(X.to_numpy().copy()).to(device)
train_y = torch.Tensor(y.to_numpy().copy()).to(device)
train_dataset = TensorDataset(train_x, train_y)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, 
                                           shuffle=True, drop_last=True)

test_x = torch.Tensor(X_test.to_numpy().copy()).to(device)
test_y = torch.Tensor(y_test.to_numpy().copy()).to(device)
test_dataset = TensorDataset(test_x, test_y)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, 
                                          shuffle=False, drop_last=True)

In [ ]:
model_call = NeuralNetwork(3, 1).to(device)

In [ ]:
call_optimizer = optim.Adam(model_call.parameters(), lr=0.001)
call_loss_fn = nn.MSELoss()

In [ ]:
history_dict['call'] = train_model(model_call, train_loader, test_loader,
                                   call_loss_fn, call_optimizer, no_epochs)

In [ ]:
result_call = model_call(test_x)

In [ ]:
call_mse = mean_squared_error(result_call.detach().cpu().numpy(), y_test.to_numpy())
call_mse

### Put Option

In [ ]:
X, y = prepareDataSingle(train_df, False)

In [ ]:
X_test, y_test = prepareDataSingle(test_df, False)

In [ ]:

standard_scalar = StandardScaler()
X[['Vol', 'Spot', 'T']] = standard_scalar.fit_transform(X[['Vol', 'Spot', 'T']])
X_test[['Vol', 'Spot', 'T']] = standard_scalar.transform(X_test[['Vol', 'Spot', 'T']])

In [ ]:
train_x = torch.Tensor(X.to_numpy()).to(device)
train_y = torch.Tensor(y.to_numpy()).to(device)
train_dataset = TensorDataset(train_x, train_y)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, 
                                           shuffle=True, drop_last=True)

test_x = torch.Tensor(X_test.to_numpy()).to(device)
test_y = torch.Tensor(y_test.to_numpy()).to(device)
test_dataset = TensorDataset(test_x, test_y)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, 
                                          shuffle=False, drop_last=True)

In [ ]:
model_put = NeuralNetwork(3, 1).to(device)

In [ ]:
put_optimizer = optim.Adam(model_put.parameters(), lr=0.001)
put_loss_fn = nn.MSELoss()

In [ ]:
history_dict['put'] = train_model(model_put, train_loader, test_loader,
                                  put_loss_fn, put_optimizer, no_epochs)

In [ ]:
result_put = model_put(test_x)

In [ ]:
put_mse = mean_squared_error(result_put.detach().cpu().numpy(), y_test.to_numpy())
put_mse

### Both

In [ ]:
X, y = prepareDataBoth(train_df)

In [ ]:
X_test, y_test = prepareDataBoth(test_df)

In [ ]:
standard_scalar = StandardScaler()
X[['Vol', 'Spot', 'T']] = standard_scalar.fit_transform(X[['Vol', 'Spot', 'T']])
X_test[['Vol', 'Spot', 'T']] = standard_scalar.transform(X_test[['Vol', 'Spot', 'T']])

In [ ]:
train_x = torch.Tensor(X.to_numpy()).to(device)
train_y = torch.Tensor(y.to_numpy()).to(device)
train_dataset = TensorDataset(train_x, train_y)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, 
                                           shuffle=True, drop_last=True)

test_x = torch.Tensor(X_test.to_numpy()).to(device)
test_y = torch.Tensor(y_test.to_numpy()).to(device)
test_dataset = TensorDataset(test_x, test_y)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, 
                                          shuffle=False, drop_last=True)

In [ ]:
model_both = NeuralNetwork(3, 2).to(device)

In [ ]:
both_optimizer = optim.Adam(model_both.parameters(), lr=0.001)
both_loss_fn = nn.MSELoss()

In [ ]:
history_dict['both'] = train_model(model_both, train_loader, test_loader,
                                   both_loss_fn, both_optimizer, no_epochs)

In [ ]:
result_both = model_both(test_x)

In [ ]:
both_mse = mean_squared_error(result_both.detach().cpu().numpy(), y_test.to_numpy())
both_mse

In [ ]:
put_both_mse = mean_squared_error(result_both[:,1].detach().cpu().numpy(), y_test['PutValue'].to_numpy())
put_both_mse

In [ ]:
call_both_mse = mean_squared_error(result_both[:,0].detach().cpu().numpy(), y_test['CallValue'].to_numpy())
call_both_mse